## Contact MinIO

In [2]:
%pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 224.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 128.0 kB/s eta 0:00:0000:0100:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 154.3 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [31]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://host.docker.internal:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
    config=boto3.session.Config(signature_version="s3v4")
)

BUCKET = "bronze"

## Reading files from MinIO

In [24]:
import socket

print(socket.gethostname())

ba2fc202e1af


In [33]:
response = s3.list_buckets()

print(response)
print(type(response))

{'ResponseMetadata': {'RequestId': '18A746AFC273D2DF', 'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'accept-ranges': 'bytes', 'content-length': '364', 'content-type': 'application/xml', 'server': 'MinIO', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'vary': 'Origin, Accept-Encoding', 'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'x-amz-request-id': '18A746AFC273D2DF', 'x-content-type-options': 'nosniff', 'x-ratelimit-limit': '3144', 'x-ratelimit-remaining': '3144', 'x-xss-protection': '1; mode=block', 'date': 'Fri, 17 Apr 2026 22:51:38 GMT'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'bronze', 'CreationDate': datetime.datetime(2026, 4, 16, 12, 11, 8, 184000, tzinfo=tzlocal())}], 'Owner': {'DisplayName': 'minio', 'ID': '02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4'}}
<class 'dict'>


In [34]:
response = s3.list_objects_v2(Bucket=BUCKET)

for obj in response.get("Contents", []):
    print(obj["Key"])

adzuna/ingestion_timestamp=1776445291/data.json
arbeitnow/ingestion_timestamp=1776341456/data.json
arbeitnow/ingestion_timestamp=1776445291/data.json
arbeitnow/ingestion_timestamp=1776462223/data.json
metadata/last_run.json
reed/ingestion_timestamp=1776445291/data.json
reed/ingestion_timestamp=1776462223/data.json


In [46]:
import json
import pandas as pd

def load_latest_source_raw(source_name):
    response = s3.list_objects_v2(Bucket=BUCKET)

    files = [
        obj["Key"] for obj in response.get("Contents", [])
        if source_name in obj["Key"] and obj["Key"].endswith("data.json")
    ]

    if not files:
        print(f"No files found for {source_name}")
        return None

    latest_file = sorted(files)[-1]

    response = s3.get_object(Bucket=BUCKET, Key=latest_file)
    data = json.loads(response["Body"].read().decode("utf-8"))

    print(f"{source_name} loaded (RAW)")

    return data
adzuna_raw = load_latest_source_raw("adzuna")
reed_raw = load_latest_source_raw("reed")
arbeitnow_raw = load_latest_source_raw("arbeitnow")

adzuna loaded (RAW)
reed loaded (RAW)
arbeitnow loaded (RAW)


In [47]:
import sys
sys.path.append(r"C:\Users\Admin\OneDrive\Desktop\Projects\JobIntelligent-Data-Platform")
from scripts.processing.cleaner import clean_adzuna_data, clean_reed_data, clean_arbeitnow_data
adzuna_clean = clean_adzuna_data(adzuna_raw)
reed_clean = clean_reed_data(reed_raw)
arbeitnow_clean = clean_arbeitnow_data(arbeitnow_raw)

/home/jovyan/scripts/processing/cleaner.py:48: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(str(text), "html.parser")


2026-04-17 23:21:30,881 | INFO | data_cleaner | [Adzuna] Initial records: 621
2026-04-17 23:21:30,885 | INFO | data_cleaner | [Adzuna] Final records: 221
2026-04-17 23:21:30,892 | INFO | data_cleaner | [Adzuna] Dropped records: 400


/home/jovyan/scripts/processing/cleaner.py:48: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(str(text), "html.parser")


TypeError: clean_reed_data() takes 0 positional arguments but 1 was given

In [38]:
print("Adzuna columns:", set(adzuna_df.columns))
print("Reed columns:", set(reed_df.columns))
print("Arbeitnow columns:", set(arbeitnow_df.columns))

Adzuna columns: {'remote', 'company_name', 'expires_date', 'source_site', 'contract_type', 'posted_date', 'job_description', 'tags', 'job_title', 'job_url', 'location', 'salary_min', 'job_id', 'salary_max', 'currency'}
Reed columns: set()
Arbeitnow columns: {'remote', 'company_name', 'job_types', 'slug', 'title', 'url', 'created_at', 'location', 'tags', 'description'}


Finding common columns

In [39]:
common_cols = set(adzuna_df.columns) & set(reed_df.columns) & set(arbeitnow_df.columns)
print(common_cols)

set()


In [40]:
print("Only in Adzuna:", set(adzuna_df.columns) - common_cols)
print("Only in Reed:", set(reed_df.columns) - common_cols)
print("Only in Arbeitnow:", set(arbeitnow_df.columns) - common_cols)

Only in Adzuna: {'remote', 'company_name', 'expires_date', 'source_site', 'contract_type', 'posted_date', 'job_description', 'tags', 'job_title', 'job_url', 'location', 'salary_min', 'job_id', 'salary_max', 'currency'}
Only in Reed: set()
Only in Arbeitnow: {'remote', 'company_name', 'job_types', 'slug', 'title', 'url', 'created_at', 'location', 'tags', 'description'}


In [41]:
def compare_basic_info(dfs, names):
    for df, name in zip(dfs, names):
        print(f"\n===== {name} =====")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print("Missing values:\n", df.isnull().sum().sort_values(ascending=False).head(10))

In [43]:
def compare_column_values(dfs, column):
    for df in dfs:
        print("\n--- SOURCE ---")
        print(df[column].value_counts().head(10))

compare_column_values([adzuna_df, reed_df, arbeitnow_df], "location")


--- SOURCE ---
location
Paris, Ile-de-France                         242
Saint-Quentin-en-Yvelines, Versailles        103
Issy-les-Moulineaux, Boulogne-Billancourt    102
8ème Arrondissement, Paris                    21
France                                        15
Seine-Maritime, Normandie                     10
Hauts-de-Seine, Ile-de-France                  6
1er Arrondissement, Paris                      6
Lyon, Rhône                                    5
Levallois-Perret, Nanterre                     5
Name: count, dtype: int64

--- SOURCE ---


KeyError: 'location'

In [ ]:
def find_differences(df1, df2, column):
    set1 = set(df1[column].dropna().unique())
    set2 = set(df2[column].dropna().unique())

    print("Only in DF1:", list(set1 - set2)[:10])
    print("Only in DF2:", list(set2 - set1)[:10])